<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/Neural-Recommendation-Personalization-Engine/blob/main/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import os
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login,HfApi
login()

KeyboardInterrupt: 

In [ ]:
# !pip install -q pandas==2.2.3
# !pip -q install -U huggingface_hub pandas pyarrow
# !pip install datasets faiss-cpu polars -q

In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/benchmark/5core/rating_only/Electronics.csv"
reviews = pd.read_csv(url)

print("Shape:", reviews.shape)
reviews.head()

Shape: (15473536, 4)


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [ ]:
print('Shape:',reviews.shape,'\n\n')
print('Columns:')
print(reviews.columns.tolist(),'\n')
print('Data Types:',reviews.dtypes,'\n')
print('Missing Values:',reviews.isna().sum())
print('\nDuplicate Rows:',reviews.duplicated().sum())
print('\n Rating Distribution',reviews['rating'].value_counts().sort_index())

print("\nUnique Users:", reviews["user_id"].nunique())
print("Unique Products:", reviews["parent_asin"].nunique())

display(reviews.head())

Shape: (15473536, 4) 


Columns:
['user_id', 'parent_asin', 'rating', 'timestamp'] 

Data Types: user_id         object
parent_asin     object
rating         float64
timestamp        int64
dtype: object 

Missing Values: user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Duplicate Rows: 0

 Rating Distribution rating
1.0     1287788
2.0      713558
3.0     1074820
4.0     2190347
5.0    10207023
Name: count, dtype: int64

Unique Users: 1641026
Unique Products: 368228


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [ ]:
shard.shape

(161001, 16)

In [3]:
meta1 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00000-of-00010.parquet")

meta2 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00001-of-00010.parquet")

meta3 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00002-of-00010.parquet")

meta4 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00003-of-00010.parquet")

meta5 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00004-of-00010.parquet")

meta6 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00005-of-00010.parquet")

meta7 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00006-of-00010.parquet")

meta8 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00007-of-00010.parquet")

meta9 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00008-of-00010.parquet")

meta10 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00009-of-00010.parquet")

In [6]:
meta1.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,None,None,None
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,None,None,None
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,None,None,None
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,None,None,None
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,None,None,None


In [7]:
meta10.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,"Samsung Galaxy Tab A 10.1 Case, Kickstand Armo...",3.6,4,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",X-Tablet,"[Electronics, Computers & Accessories, Tablet ...","{""Package Dimensions"": ""10.8 x 6.6 x 1 inches""...",B078Z6L24Z,None,None,None
1,Computers,LeiJue Keyboard Case for iPad 10.2(7th/8th/9th...,4.0,5,[Compatibility: Design for iPad 9th Generation...,[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",LeiJue,"[Electronics, Computers & Accessories, Tablet ...","{""Package Dimensions"": ""11.14 x 8.31 x 1.34 in...",B09TVKK86T,None,None,None
2,Computers,Laptop Keyboard for Gateway M360-3401701 M460A...,3.0,1,[],[],None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Gateway,"[Electronics, Computers & Accessories, Compute...","{""Manufacturer"": ""Gateway"", ""Date First Availa...",B000X56KV0,None,None,None
3,Portable Audio & Accessories,iCloth Lens and Screen Cleaner Pro-Grade Indiv...,5.0,5,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",iCloth,"[Electronics, Television & Video, Accessories,...","{""Brand Name"": ""iCloth"", ""Item Weight"": ""0.64 ...",B01K3NNEDS,None,None,None
4,Computers,Boskin for AMZ F i r e 7 case 2019 2017 Releas...,4.5,74,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Boskin,"[Electronics, Computers & Accessories, Tablet ...","{""Brand"": ""Boskin"", ""Compatible Devices"": ""Tab...",B08B1V1T2S,None,None,None
